In [1]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.preprocessing import MinMaxScaler



In [2]:
import pandas as pd
import numpy as np
import torch
from sklearn.preprocessing import MinMaxScaler

# Load the dataset
file_path = 'all_data.csv'
real_data = pd.read_csv(file_path)

# Backup original columns
original_columns = real_data.columns.tolist()

# Normalize numerical features
numerical_features = ['age', 'restingBP', 'cholestrol', 'maxheartrate', 'oldpeak']
scaler = MinMaxScaler()
real_data[numerical_features] = scaler.fit_transform(real_data[numerical_features])

# Backup the original categorical features
categorical_features = ['gender', 'chestpaintype', 'fastingbloodsugar', 'restingECG', 'exerciseangia', 'slope', 'target']

# Encode categorical features
real_data = pd.get_dummies(real_data, columns=categorical_features)

# Process the boolean columns if any
bool_cols = real_data.select_dtypes(include=['bool']).columns
if len(bool_cols) > 0:
    real_data[bool_cols] = real_data[bool_cols].astype(int)

# Fill missing values
real_data.fillna(0, inplace=True)

# Decode the one-hot encoded columns back to original categorical columns
for feature in categorical_features:
    # Find all one-hot encoded columns for the feature
    one_hot_cols = [col for col in real_data.columns if col.startswith(feature + '_')]
    if one_hot_cols:
        # Get the category with the highest probability or presence (argmax)
        real_data[feature] = real_data[one_hot_cols].idxmax(axis=1).str.split('_').str[-1].astype(int)
        # Drop the one-hot encoded columns
        real_data.drop(columns=one_hot_cols, inplace=True)

# Ensure the final columns match the original structure
real_data = real_data[original_columns]

# Convert to PyTorch tensor
data_tensor = torch.tensor(real_data.values, dtype=torch.float32)

# Verify the structure
print("Final processed data:")
print(real_data.head())
print(f"Data tensor shape: {data_tensor.shape}")
print(f"Column names: {list(real_data.columns)}")
print(f"Data types:\n{real_data.dtypes}")


Final processed data:
        age  gender  chestpaintype  restingBP  cholestrol  fastingbloodsugar  \
0  0.550000       1              2      0.855    0.000000                  0   
1  0.333333       1              0      0.470    0.379768                  0   
2  0.483333       1              2      0.665    0.235489                  0   
3  0.383333       1              0      0.690    0.489221                  1   
4  0.183333       1              1      0.995    0.000000                  0   

   restingECG  maxheartrate  exerciseangia   oldpeak  slope  target  
0           1      0.612676              0  0.897727      3       1  
1           1      0.387324              0  0.715909      1       0  
2           0      1.000000              1  0.863636      1       0  
3           1      0.654930              0  0.659091      2       1  
4           2      0.535211              0  0.897727      3       1  
Data tensor shape: torch.Size([2493, 12])
Column names: ['age', 'gender', 'ch

In [3]:

# Define GAN components: Generator and Discriminator
class Generator(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim),
            nn.Tanh()
        )

    def forward(self, x):
        return self.model(x)

class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.LeakyReLU(0.2),
            nn.Linear(256, 128),
            nn.LeakyReLU(0.2),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

In [4]:

# Model parameters
latent_dim = 100
data_dim = data_tensor.shape[1]
batch_size = 64
epochs = 5000

# Initialize models
generator = Generator(latent_dim, data_dim)
discriminator = Discriminator(data_dim)

# Optimizers and loss function
optimizer_G = optim.Adam(generator.parameters(), lr=0.0002)
optimizer_D = optim.Adam(discriminator.parameters(), lr=0.0002)
criterion = nn.BCELoss()


In [5]:

# Training GAN
for epoch in range(epochs):
    # Train Discriminator
    real_samples = data_tensor[torch.randint(0, data_tensor.size(0), (batch_size,))]
    real_labels = torch.ones((batch_size, 1))
    fake_samples = generator(torch.randn(batch_size, latent_dim))
    fake_labels = torch.zeros((batch_size, 1))
    
    discriminator_loss = criterion(discriminator(real_samples), real_labels) + \
                         criterion(discriminator(fake_samples.detach()), fake_labels)
    optimizer_D.zero_grad()
    discriminator_loss.backward()
    optimizer_D.step()

    # Train Generator
    fake_samples = generator(torch.randn(batch_size, latent_dim))
    generator_loss = criterion(discriminator(fake_samples), real_labels)
    optimizer_G.zero_grad()
    generator_loss.backward()
    optimizer_G.step()

    if epoch % 500 == 0:
        print(f"Epoch {epoch}/{epochs}, D Loss: {discriminator_loss.item()}, G Loss: {generator_loss.item()}")


Epoch 0/5000, D Loss: 1.3691658973693848, G Loss: 0.6890883445739746
Epoch 500/5000, D Loss: 0.4220454692840576, G Loss: 2.065443754196167
Epoch 1000/5000, D Loss: 0.813696563243866, G Loss: 1.5191926956176758
Epoch 1500/5000, D Loss: 0.46763065457344055, G Loss: 2.0397815704345703
Epoch 2000/5000, D Loss: 0.29930394887924194, G Loss: 2.2156076431274414
Epoch 2500/5000, D Loss: 0.31513985991477966, G Loss: 2.502685070037842
Epoch 3000/5000, D Loss: 0.3099987208843231, G Loss: 2.6895689964294434
Epoch 3500/5000, D Loss: 0.29975342750549316, G Loss: 2.618375301361084
Epoch 4000/5000, D Loss: 0.15164819359779358, G Loss: 3.2879512310028076
Epoch 4500/5000, D Loss: 0.2037499099969864, G Loss: 2.6364543437957764


In [6]:


# Generate synthetic data
num_synthetic_samples = 3000000
synthetic_data = []
for _ in range(num_synthetic_samples // batch_size):
    latent_vectors = torch.randn(batch_size, latent_dim)
    synthetic_batch = generator(latent_vectors).detach().numpy()
    synthetic_data.append(synthetic_batch)

synthetic_data = np.vstack(synthetic_data)

# Postprocessing: Denormalize numerical features
synthetic_data[:, :len(numerical_features)] = scaler.inverse_transform(
    synthetic_data[:, :len(numerical_features)]
)

# Convert back to DataFrame
synthetic_df = pd.DataFrame(synthetic_data, columns=real_data.columns)

In [7]:

# Save to CSV
output_path = 'new2.csv'
synthetic_df.to_csv(output_path, index=False)
print(f"Synthetic data saved to {output_path}")


PermissionError: [Errno 13] Permission denied

In [2]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np

# Sample dataset
data = pd.read_csv('all_data.csv')

df = pd.DataFrame(data)

# Normalize the data
data = df.values
data = (data - np.min(data, axis=0)) / (np.max(data, axis=0) - np.min(data, axis=0))

# Convert data to torch tensor
data = torch.tensor(data, dtype=torch.float32)

# Define the generator
class Generator(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(Generator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, output_dim),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

# Define the discriminator
class Discriminator(nn.Module):
    def __init__(self, input_dim):
        super(Discriminator, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

# Initialize the models
input_dim = data.shape[1]
generator = Generator(input_dim, input_dim)
discriminator = Discriminator(input_dim)

# Loss and optimizers
criterion = nn.BCELoss()
optimizer_g = torch.optim.Adam(generator.parameters(), lr=0.0002)
optimizer_d = torch.optim.Adam(discriminator.parameters(), lr=0.0002)

# Training the GAN
epochs = 10000
batch_size = 32

for epoch in range(epochs):
    # Train discriminator
    idx = np.random.randint(0, data.shape[0], batch_size)
    real_data = data[idx]
    noise = torch.randn(batch_size, input_dim)
    fake_data = generator(noise)
    
    real_labels = torch.ones(batch_size, 1)
    fake_labels = torch.zeros(batch_size, 1)
    
    optimizer_d.zero_grad()
    outputs = discriminator(real_data)
    d_loss_real = criterion(outputs, real_labels)
    outputs = discriminator(fake_data.detach())
    d_loss_fake = criterion(outputs, fake_labels)
    d_loss = d_loss_real + d_loss_fake
    d_loss.backward()
    optimizer_d.step()
    
    # Train generator
    optimizer_g.zero_grad()
    outputs = discriminator(fake_data)
    g_loss = criterion(outputs, real_labels)
    g_loss.backward()
    optimizer_g.step()
    
    if epoch % 1000 == 0:
        print(f"Epoch {epoch}, Discriminator Loss: {d_loss.item()}, Generator Loss: {g_loss.item()}")

# Generate synthetic data
noise = torch.randn(3000000, input_dim)
synthetic_data = generator(noise).detach().numpy()

# Convert synthetic data back to original scale and data types
synthetic_data = synthetic_data * (np.max(df.values, axis=0) - np.min(df.values, axis=0)) + np.min(df.values, axis=0)
synthetic_df = pd.DataFrame(synthetic_data, columns=df.columns)
synthetic_df = synthetic_df.astype(df.dtypes)

print("Synthetic Data:\n", synthetic_df.head())

# Save to CSV
output_path = 'synthetic_data.csv'
synthetic_df.to_csv(output_path, index=False)
print(f"Synthetic data saved to {output_path}")

Epoch 0, Discriminator Loss: 1.3932414054870605, Generator Loss: 0.7273055911064148
Epoch 1000, Discriminator Loss: 0.19498097896575928, Generator Loss: 2.7059874534606934
Epoch 2000, Discriminator Loss: 0.4434502124786377, Generator Loss: 2.092681407928467
Epoch 3000, Discriminator Loss: 0.2818736433982849, Generator Loss: 2.4513132572174072
Epoch 4000, Discriminator Loss: 0.20758461952209473, Generator Loss: 2.884406805038452
Epoch 5000, Discriminator Loss: 0.16930541396141052, Generator Loss: 3.480649948120117
Epoch 6000, Discriminator Loss: 0.2516227066516876, Generator Loss: 2.9195778369903564
Epoch 7000, Discriminator Loss: 0.05288996547460556, Generator Loss: 4.068927764892578
Epoch 8000, Discriminator Loss: 0.6922962665557861, Generator Loss: 1.9064233303070068
Epoch 9000, Discriminator Loss: 0.5448910593986511, Generator Loss: 2.2335352897644043
Synthetic Data:
    age  gender  chestpaintype  restingBP  cholestrol  fastingbloodsugar  \
0   46       0              2        159 

In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Load the synthetic data
synthetic_df = pd.read_csv('synthetic_data.csv').sample(n=100000)

# Split the data into features and target
X = synthetic_df.drop('target', axis=1).values
y = synthetic_df['target'].values

# Standardize the features
scaler = StandardScaler()
X = scaler.fit_transform(X)

# Split the data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Convert data to torch tensors
X_train = torch.tensor(X_train, dtype=torch.float32)
X_test = torch.tensor(X_test, dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.float32).view(-1, 1)
y_test = torch.tensor(y_test, dtype=torch.float32).view(-1, 1)

# Define the neural network model
class NeuralNetwork(nn.Module):
    def __init__(self, input_dim):
        super(NeuralNetwork, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid()
        )

    def forward(self, x):
        return self.model(x)

# Initialize the model, loss function, and optimizer
input_dim = X_train.shape[1]
model = NeuralNetwork(input_dim)
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

# Train the model
epochs = 1
batch_size = 64

for epoch in range(epochs):
    model.train()
    permutation = torch.randperm(X_train.size()[0])
    
    for i in range(0, X_train.size()[0], batch_size):
        indices = permutation[i:i+batch_size]
        batch_x, batch_y = X_train[indices], y_train[indices]
        
        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
    
    print(f"Epoch {epoch+1}/{epochs}, Loss: {loss.item()}")

# Evaluate the model
model.eval()
with torch.no_grad():
    outputs = model(X_test)
    predicted = (outputs > 0.5).float()
    accuracy = (predicted == y_test).float().mean()
    print(f"Accuracy: {accuracy.item() * 100:.2f}%")

Epoch 1/1, Loss: 0.03288015350699425
Accuracy: 96.88%


In [12]:
import pandas as pd 
d = pd.read_csv('synthetic_data.csv')
d.info()
d.drop_duplicates()
d.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000000 entries, 0 to 2999999
Data columns (total 12 columns):
 #   Column             Dtype  
---  ------             -----  
 0   age                int64  
 1   gender             int64  
 2   chestpaintype      int64  
 3   restingBP          int64  
 4   cholestrol         int64  
 5   fastingbloodsugar  int64  
 6   restingECG         int64  
 7   maxheartrate       int64  
 8   exerciseangia      int64  
 9   oldpeak            float64
 10  slope              int64  
 11  target             int64  
dtypes: float64(1), int64(11)
memory usage: 274.7 MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000000 entries, 0 to 2999999
Data columns (total 12 columns):
 #   Column             Dtype  
---  ------             -----  
 0   age                int64  
 1   gender             int64  
 2   chestpaintype      int64  
 3   restingBP          int64  
 4   cholestrol         int64  
 5   fastingbloodsugar  int64  
 6   restingECG  